# ETL Gold - training_dataset_v0

Construye una tabla diaria lista para entrenamiento de modelos ML para `codigoestacao = 74100000`.

In [ ]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

LEVEL_TABLE = 'weather.silver.river_levels_daily'
TEMP_TABLE = 'weather.silver.temperature_daily'
RAIN_TABLE = 'weather.silver.rainfall_daily'
DISCHARGE_TABLE = 'weather.silver.river_discharge_daily'
SUBCUENCA_TABLE = 'weather.silver.estacion_subcuenca'
QUALITY_TABLE = 'weather.silver.attribute_quality'
TARGET_TABLE = 'weather.gold.training_dataset_v0'
TARGET_STATION = '74100000'
PUNTO_PREDICCION = 'ana_74100000'
SUBCUENCAS = ['alta_frontera', 'intermedia_paso_libres', 'baja_salto_grande']

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def quality_is_usable(source_table, attribute_name):
    rows = (
        spark.table(QUALITY_TABLE)
        .filter(F.col('source_table') == F.lit(source_table))
        .filter(F.col('attribute_name') == F.lit(attribute_name))
        .filter(F.col('grain') == F.lit('global_source_daily'))
        .orderBy(F.col('evaluated_at').desc_nulls_last())
        .limit(1)
        .collect()
    )
    if not rows:
        return False
    return bool(rows[0]['is_usable'])


def build_calendar(levels_df):
    bounds = levels_df.agg(F.min('fecha').alias('min_fecha'), F.max('fecha').alias('max_fecha')).first()
    if bounds['min_fecha'] is None or bounds['max_fecha'] is None:
        raise ValueError('No level data available for Gold calendar')

    return spark.sql(
        f"SELECT explode(sequence(to_date('{bounds['min_fecha']}'), to_date('{bounds['max_fecha']}'), interval 1 day)) AS fecha"
    )


def target_window(dataset_df):
    source_bounds = dataset_df.agg(F.min('fecha').alias('min_fecha'), F.max('fecha').alias('max_fecha')).first()
    if source_bounds['min_fecha'] is None or source_bounds['max_fecha'] is None:
        raise ValueError('No Gold rows generated')

    if load_mode == 'full':
        return source_bounds['min_fecha'], source_bounds['max_fecha']

    max_target_fecha = (
        spark.table(TARGET_TABLE)
        .filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))
        .agg(F.max('fecha').alias('max_fecha'))
        .first()['max_fecha']
    )
    if max_target_fecha is None:
        return source_bounds['min_fecha'], source_bounds['max_fecha']

    changed_min = max(source_bounds['min_fecha'], max_target_fecha - timedelta(days=incremental_lookback_days))
    changed_max = source_bounds['max_fecha']
    window_start = max(source_bounds['min_fecha'], changed_min - timedelta(days=14))
    window_end = min(source_bounds['max_fecha'], changed_max + timedelta(days=7))
    return window_start, window_end


def replace_target_window(dataset_df, window_start, window_end):
    target_delta = DeltaTable.forName(spark, TARGET_TABLE)
    if load_mode == 'full':
        delete_condition = f"punto_prediccion = '{PUNTO_PREDICCION}'"
    else:
        delete_condition = f"punto_prediccion = '{PUNTO_PREDICCION}' AND fecha >= DATE '{window_start}' AND fecha <= DATE '{window_end}'"

    print(f'Deleting Gold rows with condition: {delete_condition}')
    target_delta.delete(delete_condition)

    rows_to_write = dataset_df.filter((F.col('fecha') >= F.lit(window_start)) & (F.col('fecha') <= F.lit(window_end)))
    count_rows = rows_to_write.count()
    print(f'Appending {count_rows} Gold rows')
    if count_rows > 0:
        rows_to_write.write.format('delta').mode('append').saveAsTable(TARGET_TABLE)

In [ ]:
levels = (
    spark.table(LEVEL_TABLE)
    .filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
    .select(
        'fecha',
        'codigoestacao',
        F.col('nivel_media_cm').alias('nivel_rio_actual_cm'),
        F.col('nivel_media_m').alias('nivel_rio_actual_m'),
        F.col('registros_validos').alias('nivel_registros_validos'),
    )
)

calendar = build_calendar(levels)

temp_global = (
    spark.table(TEMP_TABLE)
    .groupBy('fecha')
    .agg(
        F.avg('temp_media_c').alias('temp_media_c'),
        F.min('temp_min_c').alias('temp_min_c'),
        F.max('temp_max_c').alias('temp_max_c'),
        F.countDistinct('icao_id').cast('bigint').alias('temp_station_count'),
    )
)

for attribute_name in ['temp_media_c', 'temp_min_c', 'temp_max_c']:
    if not quality_is_usable(TEMP_TABLE, attribute_name):
        print(f'{attribute_name} is not usable; publishing NULLs')
        temp_global = temp_global.withColumn(attribute_name, F.lit(None).cast('double'))

rain_usable = quality_is_usable(RAIN_TABLE, 'lluvia_acumulada_mm')
if rain_usable:
    rain_global = spark.table(RAIN_TABLE).groupBy('fecha').agg(F.sum('lluvia_acumulada_mm').alias('lluvia_acumulada_mm'))
else:
    rain_global = calendar.select('fecha').withColumn('lluvia_acumulada_mm', F.lit(None).cast('double'))

# Caudal de la estacion target (Decision D2: caudal es el target principal, ver
# docs/rating_curve_discharge_plan.md Seccion 1 y 4.5). El nivel se mantiene intacto arriba.
discharge_usable = quality_is_usable(DISCHARGE_TABLE, 'caudal_m3s')
discharge_target = (
    spark.table(DISCHARGE_TABLE)
    .filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
    .select(
        'fecha',
        F.col('caudal_m3s').alias('caudal_actual_m3s'),
        'caudal_metodo', 'caudal_extrapolado', 'distancia_fuera_rango_cm', 'supera_aforo_maximo', 'caudal_confiable',
    )
)

# Agregados de caudal por sub-cuenca (SIG/subcuencas_modelo.geojson via weather.silver.estacion_subcuenca):
# el caudal es fisicamente aditivo entre estaciones (el nivel no), asi que sumar el caudal
# de las estaciones de una sub-cuenca da el aporte total aguas arriba de ese punto. Hoy
# solo el grupo A (22 estaciones, todas en alta_frontera) esta mapeado -- las columnas de
# intermedia_paso_libres/baja_salto_grande quedan en NULL hasta que el grupo B (Fase 1,
# ver plan) tenga curva y mapeo de sub-cuenca.
subcuenca_daily = (
    spark.table(DISCHARGE_TABLE).alias('d')
    .join(spark.table(SUBCUENCA_TABLE).alias('sc'), 'codigoestacao', 'inner')
    .groupBy('fecha', 'subcuenca')
    .agg(
        F.sum('caudal_m3s').alias('caudal_agregado_m3s'),
        F.avg(F.col('caudal_confiable').cast('double')).alias('confiable_pct'),
    )
)

subcuenca_wide = calendar.select('fecha')
for subcuenca in SUBCUENCAS:
    one = (
        subcuenca_daily.filter(F.col('subcuenca') == F.lit(subcuenca))
        .select(
            'fecha',
            F.col('caudal_agregado_m3s').alias(f'caudal_agregado_{subcuenca}_m3s'),
            F.col('confiable_pct').alias(f'caudal_agregado_{subcuenca}_confiable_pct'),
        )
    )
    subcuenca_wide = subcuenca_wide.join(one, 'fecha', 'left')

base = (
    calendar.join(levels, 'fecha', 'left')
    .join(temp_global, 'fecha', 'left')
    .join(rain_global, 'fecha', 'left')
    .join(discharge_target, 'fecha', 'left')
    .join(subcuenca_wide, 'fecha', 'left')
    .withColumn('punto_prediccion', F.lit(PUNTO_PREDICCION))
    .withColumn('codigoestacao', F.lit(TARGET_STATION))
    .withColumn('lluvia_is_usable', F.lit(rain_usable))
    .withColumn('caudal_registros_validos', F.when(F.col('caudal_actual_m3s').isNotNull(), F.lit(1)).otherwise(F.lit(0)).cast('bigint'))
)

if not discharge_usable:
    print('caudal_m3s is not usable; publishing NULLs for caudal_actual_m3s')
    base = base.withColumn('caudal_actual_m3s', F.lit(None).cast('double'))

w = Window.orderBy('fecha')
w3 = w.rowsBetween(-2, 0)
w7 = w.rowsBetween(-6, 0)

dataset = (
    base.withColumn('nivel_rio_lag_1d', F.lag('nivel_rio_actual_m', 1).over(w))
    .withColumn('nivel_rio_lag_3d', F.lag('nivel_rio_actual_m', 3).over(w))
    .withColumn('nivel_rio_lag_7d', F.lag('nivel_rio_actual_m', 7).over(w))
    .withColumn('nivel_rio_media_3d', F.avg('nivel_rio_actual_m').over(w3))
    .withColumn('nivel_rio_media_7d', F.avg('nivel_rio_actual_m').over(w7))
    .withColumn('nivel_rio_delta_1d', F.col('nivel_rio_actual_m') - F.col('nivel_rio_lag_1d'))
    .withColumn('nivel_rio_t_mas_1d', F.lead('nivel_rio_actual_m', 1).over(w))
    .withColumn('nivel_rio_t_mas_3d', F.lead('nivel_rio_actual_m', 3).over(w))
    .withColumn('nivel_rio_t_mas_7d', F.lead('nivel_rio_actual_m', 7).over(w))
    .withColumn('nivel_rio_t_mas_14d', F.lead('nivel_rio_actual_m', 14).over(w))
    .withColumn('caudal_lag_1d', F.lag('caudal_actual_m3s', 1).over(w))
    .withColumn('caudal_lag_3d', F.lag('caudal_actual_m3s', 3).over(w))
    .withColumn('caudal_lag_7d', F.lag('caudal_actual_m3s', 7).over(w))
    .withColumn('caudal_media_3d', F.avg('caudal_actual_m3s').over(w3))
    .withColumn('caudal_media_7d', F.avg('caudal_actual_m3s').over(w7))
    .withColumn('caudal_delta_1d', F.col('caudal_actual_m3s') - F.col('caudal_lag_1d'))
    .withColumn('caudal_t_mas_1d', F.lead('caudal_actual_m3s', 1).over(w))
    .withColumn('caudal_t_mas_3d', F.lead('caudal_actual_m3s', 3).over(w))
    .withColumn('caudal_t_mas_7d', F.lead('caudal_actual_m3s', 7).over(w))
    .withColumn('caudal_t_mas_14d', F.lead('caudal_actual_m3s', 14).over(w))
    .withColumn('feature_generated_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
)

for subcuenca in SUBCUENCAS:
    col_m3s = f'caudal_agregado_{subcuenca}_m3s'
    dataset = (
        dataset
        .withColumn(f'caudal_agregado_{subcuenca}_lag_1d', F.lag(col_m3s, 1).over(w))
        .withColumn(f'caudal_agregado_{subcuenca}_lag_2d', F.lag(col_m3s, 2).over(w))
        .withColumn(f'caudal_agregado_{subcuenca}_lag_3d', F.lag(col_m3s, 3).over(w))
    )

output_columns = [
    'fecha', 'punto_prediccion', 'codigoestacao', 'nivel_rio_actual_cm', 'nivel_rio_actual_m', 'nivel_registros_validos',
    'temp_media_c', 'temp_min_c', 'temp_max_c', 'temp_station_count', 'lluvia_acumulada_mm', 'lluvia_is_usable',
    'nivel_rio_lag_1d', 'nivel_rio_lag_3d', 'nivel_rio_lag_7d', 'nivel_rio_media_3d', 'nivel_rio_media_7d', 'nivel_rio_delta_1d',
    'nivel_rio_t_mas_1d', 'nivel_rio_t_mas_3d', 'nivel_rio_t_mas_7d', 'nivel_rio_t_mas_14d',
    'caudal_actual_m3s', 'caudal_registros_validos', 'caudal_metodo', 'caudal_extrapolado', 'distancia_fuera_rango_cm',
    'supera_aforo_maximo', 'caudal_confiable', 'caudal_lag_1d', 'caudal_lag_3d', 'caudal_lag_7d',
    'caudal_media_3d', 'caudal_media_7d', 'caudal_delta_1d',
    'caudal_t_mas_1d', 'caudal_t_mas_3d', 'caudal_t_mas_7d', 'caudal_t_mas_14d',
]
for subcuenca in SUBCUENCAS:
    output_columns += [
        f'caudal_agregado_{subcuenca}_m3s', f'caudal_agregado_{subcuenca}_lag_1d',
        f'caudal_agregado_{subcuenca}_lag_2d', f'caudal_agregado_{subcuenca}_lag_3d',
        f'caudal_agregado_{subcuenca}_confiable_pct',
    ]
output_columns += ['feature_generated_at', 'updated_at']

dataset = dataset.select(*output_columns)

window_start, window_end = target_window(dataset)
print(f'Gold replacement window: {window_start} to {window_end}')
replace_target_window(dataset, window_start, window_end)

spark.table(TARGET_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows')).show()